# Advanced Python Pipelines and Broadcasting — Problems with Complete Solutions

This notebook develops production-minded, generator/coroutine-based data pipelines from first principles. It expands the classic CSV car-routing example into a self-contained lab with deterministic data, advanced routing, bounded state, lifecycle management, metrics, atomic file output, safe configuration, and asynchronous fan-out.

## Best-practice goals

- Stream records instead of loading an entire file by default.
- Separate parsing, transformation, filtering, routing, and persistence.
- Make resource ownership and shutdown explicit.
- Validate at boundaries and preserve rejected data through side outputs.
- Use bounded state for long-running pipelines.
- Prefer registries over `eval` for configuration-driven behavior.
- Test every stage with executable assertions.
- Keep examples deterministic and standard-library-only.

> Run the notebook from top to bottom. Every problem is followed by a complete solution and verification cells.

## Contents

1. Deterministic lab dataset
2. Problem 1 — Streaming validation with a dead-letter side channel
3. Problem 2 — Composable coroutine stages and close propagation
4. Problem 3 — Multi-stage transformation and broadcasting
5. Problem 4 — Dynamic rule routing (`all` versus `first` match)
6. Problem 5 — Bounded-memory deduplication
7. Problem 6 — Batching and partial-batch flushing
8. Problem 7 — Fault-isolated broadcasting
9. Problem 8 — Instrumentation and stage metrics
10. Problem 9 — Atomic CSV sinks with rollback
11. Problem 10 — Safe configuration-driven pipeline construction
12. Problem 11 — Asynchronous broadcasting with backpressure
13. Capstone — Validated, deduplicated, measured, multi-route pipeline
14. Additional practice problems

## 0. Setup and deterministic sample data

In [2]:
from __future__ import annotations

import asyncio
import csv
import json
import os
import random
import tempfile
import time
from collections import Counter, OrderedDict
from contextlib import ExitStack, contextmanager
from dataclasses import asdict, dataclass, field
from functools import wraps
from pathlib import Path
from typing import Any, Callable, Generator, Iterable, Iterator, Mapping, MutableSequence, Sequence, TypeVar

WORKDIR = Path("pipeline_lab_output")
WORKDIR.mkdir(exist_ok=True)
DATA_PATH = WORKDIR / "car_data.csv"

RANDOM_SEED = 20260721
random.seed(RANDOM_SEED)

# print(f"Working directory: {WORKDIR.resolve()}")

In [3]:
MAKES = [
    "Ford", "Toyota", "Mercedes-Benz", "BMW", "Honda",
    "Chevrolet", "Tesla", "Volvo", "Subaru", "Mazda",
]
MODELS = {
    "Ford": ["F-150", "Mustang", "Focus"],
    "Toyota": ["Camry", "RAV4", "Corolla"],
    "Mercedes-Benz": ["C-Class", "S-Class", "GLC"],
    "BMW": ["3 Series", "X5", "Z4"],
    "Honda": ["Civic", "Accord", "CR-V"],
    "Chevrolet": ["Malibu", "Tahoe", "Corvette"],
    "Tesla": ["Model 3", "Model S", "Model Y"],
    "Volvo": ["S60", "XC60", "XC90"],
    "Subaru": ["Outback", "Forester", "Impreza"],
    "Mazda": ["Mazda3", "CX-5", "MX-5"],
}
COLORS = ["Pink", "Green", "Blue", "Red", "Black", "White", "Silver"]


def synthetic_vin(index: int) -> str:
    """Return a deterministic 17-character VIN-like identifier."""
    return f"VIN{index:014d}"


def create_lab_csv(path: Path, n_valid_rows: int = 240) -> None:
    """Create valid rows, deliberate duplicates, and malformed rows."""
    fieldnames = ["car_make", "car_model", "model_year", "vin", "color"]
    rows: list[dict[str, str]] = []

    for i in range(n_valid_rows):
        make = random.choice(MAKES)
        model = random.choice(MODELS[make])
        year = random.randint(1960, 2024)
        color = random.choice(COLORS)

        # Every 45th record reuses a recent VIN so the dedupe problem has data.
        vin_index = i - 7 if i > 7 and i % 45 == 0 else i
        rows.append({
            "car_make": make,
            "car_model": model,
            "model_year": str(year),
            "vin": synthetic_vin(vin_index),
            "color": color,
        })

    malformed = [
        {"car_make": "Ford", "car_model": "Focus", "model_year": "not-a-year", "vin": synthetic_vin(9001), "color": "Blue"},
        {"car_make": "", "car_model": "Mystery", "model_year": "2019", "vin": synthetic_vin(9002), "color": "Black"},
        {"car_make": "Toyota", "car_model": "Camry", "model_year": "2035", "vin": synthetic_vin(9003), "color": "Silver"},
        {"car_make": "Honda", "car_model": "Civic", "model_year": "2018", "vin": "SHORT", "color": "Red"},
    ]

    with path.open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows + malformed)


create_lab_csv(DATA_PATH)
print(f"Created {DATA_PATH} ({DATA_PATH.stat().st_size:,} bytes)")

Created pipeline_lab_output\car_data.csv (10,549 bytes)


In [4]:
with DATA_PATH.open(newline="", encoding="utf-8") as file:
    preview = list(csv.DictReader(file))[:5]

preview

[{'car_make': 'Mazda',
  'car_model': 'Mazda3',
  'model_year': '2008',
  'vin': 'VIN00000000000000',
  'color': 'White'},
 {'car_make': 'Tesla',
  'car_model': 'Model S',
  'model_year': '1995',
  'vin': 'VIN00000000000001',
  'color': 'Silver'},
 {'car_make': 'Volvo',
  'car_model': 'S60',
  'model_year': '1986',
  'vin': 'VIN00000000000002',
  'color': 'Silver'},
 {'car_make': 'Volvo',
  'car_model': 'XC90',
  'model_year': '1976',
  'vin': 'VIN00000000000003',
  'color': 'White'},
 {'car_make': 'Mercedes-Benz',
  'car_model': 'C-Class',
  'model_year': '2009',
  'vin': 'VIN00000000000004',
  'color': 'Black'}]

## Problem 1 — Streaming validation with a dead-letter side channel

Build a parser that:

1. Reads the CSV lazily.
2. Converts each valid row into an immutable typed record.
3. Rejects missing values, impossible years, and malformed VINs.
4. Sends rejected rows to a separate error target without stopping the valid stream.
5. Includes row number, reason, and original row in every error record.

**Why this is advanced:** robust pipelines preserve bad data for diagnosis instead of silently dropping it or terminating the whole job.

### Solution 1

In [5]:
@dataclass(frozen=True, slots=True)
class CarRecord:
    make: str
    model: str
    year: int
    vin: str
    color: str


@dataclass(frozen=True, slots=True)
class ParseIssue:
    row_number: int
    reason: str
    raw_row: Mapping[str, str]


def parse_car_row(raw: Mapping[str, str], row_number: int) -> CarRecord:
    """Validate and convert one CSV row; raise ValueError on invalid input."""
    required = ("car_make", "car_model", "model_year", "vin", "color")
    missing = [name for name in required if not raw.get(name, "").strip()]
    if missing:
        raise ValueError(f"missing required field(s): {', '.join(missing)}")

    try:
        year = int(raw["model_year"])
    except ValueError as exc:
        raise ValueError("model_year must be an integer") from exc

    if not 1886 <= year <= 2026:
        raise ValueError("model_year must be between 1886 and 2026")

    vin = raw["vin"].strip().upper()
    if len(vin) != 17 or not vin.isalnum():
        raise ValueError("vin must contain exactly 17 alphanumeric characters")

    return CarRecord(
        make=raw["car_make"].strip(),
        model=raw["car_model"].strip(),
        year=year,
        vin=vin,
        color=raw["color"].strip(),
    )


def stream_car_records(
    path: Path,
    *,
    error_target: Generator[Any, ParseIssue, Any] | None = None,
) -> Iterator[CarRecord]:
    """Yield valid records lazily and optionally send invalid rows elsewhere."""
    with path.open(newline="", encoding="utf-8") as file:
        reader = csv.DictReader(file)
        for row_number, raw in enumerate(reader, start=2):
            try:
                yield parse_car_row(raw, row_number)
            except ValueError as exc:
                if error_target is not None:
                    error_target.send(ParseIssue(row_number, str(exc), dict(raw)))

In [6]:
T = TypeVar("T")


def coroutine(function: Callable[..., Generator[Any, Any, Any]]) -> Callable[..., Generator[Any, Any, Any]]:
    """Decorator that primes a generator-based coroutine exactly once."""
    @wraps(function)
    def start(*args: Any, **kwargs: Any) -> Generator[Any, Any, Any]:
        generator = function(*args, **kwargs)
        next(generator)
        return generator
    return start


@coroutine
def append_sink(buffer: MutableSequence[T]) -> Generator[None, T, None]:
    """Append every received item to a caller-owned mutable sequence."""
    while True:
        buffer.append((yield))


issues: list[ParseIssue] = []
issue_sink = append_sink(issues)
valid_records = list(stream_car_records(DATA_PATH, error_target=issue_sink))
issue_sink.close()

assert len(valid_records) == 240
assert len(issues) == 4
assert all(isinstance(record.year, int) for record in valid_records)
assert all(len(record.vin) == 17 for record in valid_records)

print(f"Valid records: {len(valid_records)}")
print(f"Rejected rows: {len(issues)}")
for issue in issues:
    print(f"row {issue.row_number}: {issue.reason}")

Valid records: 240
Rejected rows: 4
row 242: model_year must be an integer
row 243: missing required field(s): car_make
row 244: model_year must be between 1886 and 2026
row 245: vin must contain exactly 17 alphanumeric characters


### Extra example — strict mode versus tolerant mode

Sometimes a batch job should continue after bad rows; sometimes it should fail immediately. Keep that choice at the orchestration boundary rather than hard-coding it deep inside the parser.

In [7]:
def load_records(path: Path, *, strict: bool) -> tuple[list[CarRecord], list[ParseIssue]]:
    rejected: list[ParseIssue] = []
    sink = append_sink(rejected)
    records = list(stream_car_records(path, error_target=sink))
    sink.close()

    if strict and rejected:
        first = rejected[0]
        raise ValueError(f"strict load failed at row {first.row_number}: {first.reason}")
    return records, rejected


tolerant_records, tolerant_issues = load_records(DATA_PATH, strict=False)
assert len(tolerant_records) == 240 and len(tolerant_issues) == 4

## Problem 2 — Composable coroutine stages and close propagation

Implement reusable stages for:

- mapping one item to another,
- filtering items,
- broadcasting each item to multiple targets,
- closing downstream stages when the upstream stage closes.

The stage that creates or owns a downstream stage must define a clear shutdown policy. In this notebook, every composite stage closes its direct children.

### Solution 2

In [8]:
def close_unique(targets: Iterable[Generator[Any, Any, Any]]) -> None:
    """Close each distinct target once, even if it appears multiple times."""
    seen_ids: set[int] = set()
    for target in targets:
        identity = id(target)
        if identity not in seen_ids:
            seen_ids.add(identity)
            target.close()


@coroutine
def map_stage(
    transform: Callable[[T], Any],
    target: Generator[Any, Any, Any],
) -> Generator[None, T, None]:
    try:
        while True:
            target.send(transform((yield)))
    finally:
        target.close()


@coroutine
def filter_stage(
    predicate: Callable[[T], bool],
    target: Generator[Any, T, Any],
) -> Generator[None, T, None]:
    try:
        while True:
            item = yield
            if predicate(item):
                target.send(item)
    finally:
        target.close()


@coroutine
def broadcast_stage(
    targets: Sequence[Generator[Any, T, Any]],
) -> Generator[None, T, None]:
    owned_targets = tuple(targets)
    try:
        while True:
            item = yield
            for target in owned_targets:
                target.send(item)
    finally:
        close_unique(owned_targets)

In [9]:
pink: list[CarRecord] = []
recent: list[CarRecord] = []

pipeline = broadcast_stage([
    filter_stage(lambda car: car.color == "Pink", append_sink(pink)),
    filter_stage(lambda car: car.year >= 2020, append_sink(recent)),
])

for record in valid_records:
    pipeline.send(record)
pipeline.close()

assert all(car.color == "Pink" for car in pink)
assert all(car.year >= 2020 for car in recent)
assert len(pink) == sum(car.color == "Pink" for car in valid_records)
assert len(recent) == sum(car.year >= 2020 for car in valid_records)

print({"pink": len(pink), "recent": len(recent)})

{'pink': 34, 'recent': 24}


## Problem 3 — Multi-stage transformation and broadcasting

Construct a pipeline that:

1. Normalizes text fields.
2. Converts records into serializable dictionaries.
3. Broadcasts dictionaries into three independent views:
   - all pink cars,
   - green Ford cars,
   - cars older than 1990.
4. Verifies that an item matching multiple rules reaches multiple outputs.

### Solution 3

In [10]:
def normalize_car(car: CarRecord) -> CarRecord:
    return CarRecord(
        make=" ".join(car.make.split()),
        model=" ".join(car.model.split()),
        year=car.year,
        vin=car.vin.upper(),
        color=car.color.title(),
    )


def car_to_dict(car: CarRecord) -> dict[str, Any]:
    return asdict(car)


pink_rows: list[dict[str, Any]] = []
ford_green_rows: list[dict[str, Any]] = []
older_rows: list[dict[str, Any]] = []

fan_out = broadcast_stage([
    filter_stage(lambda row: row["color"] == "Pink", append_sink(pink_rows)),
    filter_stage(
        lambda row: row["make"] == "Ford" and row["color"] == "Green",
        append_sink(ford_green_rows),
    ),
    filter_stage(lambda row: row["year"] < 1990, append_sink(older_rows)),
])

pipeline = map_stage(normalize_car, map_stage(car_to_dict, fan_out))
for record in valid_records:
    pipeline.send(record)
pipeline.close()

assert all(row["color"] == "Pink" for row in pink_rows)
assert all(row["make"] == "Ford" and row["color"] == "Green" for row in ford_green_rows)
assert all(row["year"] < 1990 for row in older_rows)

multi_match_vins = {
    row["vin"] for row in pink_rows
} & {
    row["vin"] for row in older_rows
}

print({
    "pink": len(pink_rows),
    "ford_green": len(ford_green_rows),
    "older_than_1990": len(older_rows),
    "pink_and_old_overlap": len(multi_match_vins),
})

{'pink': 34, 'ford_green': 2, 'older_than_1990': 102, 'pink_and_old_overlap': 14}


## Problem 4 — Dynamic rule routing (`all` versus `first` match)

A fixed broadcaster becomes repetitive when routes are data-driven. Create a router with named rules and two modes:

- `all`: send an item to every matching route.
- `first`: send an item only to the first matching route.

Also support an unmatched target and reject invalid modes early.

### Solution 4

In [11]:
@dataclass(frozen=True)
class Route:
    name: str
    predicate: Callable[[T], bool]
    target: Generator[Any, T, Any]


@coroutine
def route_stage(
    routes: Sequence[Route],
    *,
    mode: str = "all",
    unmatched_target: Generator[Any, T, Any] | None = None,
) -> Generator[None, T, None]:
    if mode not in {"all", "first"}:
        raise ValueError("mode must be 'all' or 'first'")

    owned_targets = [route.target for route in routes]
    if unmatched_target is not None:
        owned_targets.append(unmatched_target)

    try:
        while True:
            item = yield
            matched = False
            for route in routes:
                if route.predicate(item):
                    route.target.send(item)
                    matched = True
                    if mode == "first":
                        break
            if not matched and unmatched_target is not None:
                unmatched_target.send(item)
    finally:
        close_unique(owned_targets)

In [12]:
all_pink: list[CarRecord] = []
all_old: list[CarRecord] = []
all_unmatched: list[CarRecord] = []

router_all = route_stage(
    [
        Route("pink", lambda car: car.color == "Pink", append_sink(all_pink)),
        Route("old", lambda car: car.year < 1990, append_sink(all_old)),
    ],
    mode="all",
    unmatched_target=append_sink(all_unmatched),
)
for car in valid_records:
    router_all.send(car)
router_all.close()

first_pink: list[CarRecord] = []
first_old: list[CarRecord] = []
first_unmatched: list[CarRecord] = []
router_first = route_stage(
    [
        Route("pink", lambda car: car.color == "Pink", append_sink(first_pink)),
        Route("old", lambda car: car.year < 1990, append_sink(first_old)),
    ],
    mode="first",
    unmatched_target=append_sink(first_unmatched),
)
for car in valid_records:
    router_first.send(car)
router_first.close()

assert len(all_pink) == len(first_pink)
assert len(first_old) <= len(all_old)
assert len(first_pink) + len(first_old) + len(first_unmatched) == len(valid_records)
assert len(all_pink) + len(all_old) >= len(valid_records) - len(all_unmatched)

print({
    "all_mode_deliveries": len(all_pink) + len(all_old),
    "first_mode_deliveries": len(first_pink) + len(first_old),
    "first_mode_partition_size": len(first_pink) + len(first_old) + len(first_unmatched),
})

{'all_mode_deliveries': 136, 'first_mode_deliveries': 122, 'first_mode_partition_size': 240}


## Problem 5 — Bounded-memory deduplication

A long-running stream cannot keep every historical key forever. Build an LRU-style deduplication stage that:

- drops recently seen keys,
- remembers at most `max_seen` keys,
- evicts the least recently used key,
- exposes statistics for received, emitted, dropped, and evicted items.

Explain the trade-off: after eviction, an old duplicate may pass through again.

### Solution 5

In [13]:
@dataclass
class DedupeStats:
    received: int = 0
    emitted: int = 0
    dropped: int = 0
    evicted: int = 0


@coroutine
def dedupe_stage(
    key: Callable[[T], Any],
    target: Generator[Any, T, Any],
    *,
    max_seen: int,
    stats: DedupeStats | None = None,
) -> Generator[None, T, None]:
    if max_seen <= 0:
        raise ValueError("max_seen must be positive")

    stats = stats if stats is not None else DedupeStats()
    seen: OrderedDict[Any, None] = OrderedDict()

    try:
        while True:
            item = yield
            stats.received += 1
            item_key = key(item)

            if item_key in seen:
                seen.move_to_end(item_key)
                stats.dropped += 1
                continue

            seen[item_key] = None
            if len(seen) > max_seen:
                seen.popitem(last=False)
                stats.evicted += 1

            stats.emitted += 1
            target.send(item)
    finally:
        target.close()

In [14]:
deduped: list[CarRecord] = []
dedupe_stats = DedupeStats()
deduper = dedupe_stage(
    key=lambda car: car.vin,
    target=append_sink(deduped),
    max_seen=500,
    stats=dedupe_stats,
)
for car in valid_records:
    deduper.send(car)
deduper.close()

expected_unique_vins = len({car.vin for car in valid_records})
assert len(deduped) == expected_unique_vins
assert dedupe_stats.dropped == len(valid_records) - expected_unique_vins
assert len({car.vin for car in deduped}) == len(deduped)

print(dedupe_stats)

DedupeStats(received=240, emitted=235, dropped=5, evicted=0)


### Extra example — demonstrate bounded-memory eviction

In [15]:
small_output: list[str] = []
small_stats = DedupeStats()
small_deduper = dedupe_stage(
    key=lambda value: value,
    target=append_sink(small_output),
    max_seen=3,
    stats=small_stats,
)

for value in ["A", "B", "C", "A", "D", "B"]:
    small_deduper.send(value)
small_deduper.close()

# "A" is dropped while remembered. "B" is emitted later because it was evicted.
assert small_output == ["A", "B", "C", "D", "B"]
assert small_stats.dropped == 1
assert small_stats.evicted == 2
small_output, small_stats

(['A', 'B', 'C', 'D', 'B'],
 DedupeStats(received=6, emitted=5, dropped=1, evicted=2))

## Problem 6 — Batching and partial-batch flushing

Create a stage that groups items into fixed-size tuples. It must:

- emit full batches immediately,
- flush the final partial batch when closed,
- reject non-positive batch sizes,
- close its downstream target exactly once.

This pattern is useful for bulk database inserts, API requests, and vectorized processing.

### Solution 6

In [16]:
@coroutine
def batch_stage(
    batch_size: int,
    target: Generator[Any, tuple[T, ...], Any],
) -> Generator[None, T, None]:
    if batch_size <= 0:
        raise ValueError("batch_size must be positive")

    batch: list[T] = []
    try:
        while True:
            batch.append((yield))
            if len(batch) == batch_size:
                target.send(tuple(batch))
                batch.clear()
    except GeneratorExit:
        if batch:
            target.send(tuple(batch))
    finally:
        target.close()

In [17]:
batches: list[tuple[CarRecord, ...]] = []
batcher = batch_stage(32, append_sink(batches))
for car in valid_records[:70]:
    batcher.send(car)
batcher.close()

assert [len(batch) for batch in batches] == [32, 32, 6]
assert sum(map(len, batches)) == 70
print([len(batch) for batch in batches])

[32, 32, 6]


### Extra example — batch-level aggregation

The downstream stage receives whole tuples, so expensive aggregation can occur once per batch instead of once per item.

In [18]:
batch_summaries: list[dict[str, Any]] = []
summary_sink = append_sink(batch_summaries)


def summarize_batch(batch: tuple[CarRecord, ...]) -> dict[str, Any]:
    makes = Counter(car.make for car in batch)
    return {
        "size": len(batch),
        "oldest_year": min(car.year for car in batch),
        "top_make": makes.most_common(1)[0],
    }

summary_pipeline = batch_stage(50, map_stage(summarize_batch, summary_sink))
for car in valid_records[:120]:
    summary_pipeline.send(car)
summary_pipeline.close()

assert [summary["size"] for summary in batch_summaries] == [50, 50, 20]
batch_summaries

[{'size': 50, 'oldest_year': 1961, 'top_make': ('Volvo', 9)},
 {'size': 50, 'oldest_year': 1960, 'top_make': ('Mercedes-Benz', 9)},
 {'size': 20, 'oldest_year': 1961, 'top_make': ('Tesla', 6)}]

## Problem 7 — Fault-isolated broadcasting

A normal broadcaster is fail-fast: one broken consumer stops the entire pipeline. Add a configurable failure policy:

- `fail_fast`: re-raise the first target error.
- `isolate`: remove the failed target and continue delivering to healthy targets.

Record the target name, item representation, and exception message in a side output.

### Solution 7

In [19]:
@dataclass(frozen=True)
class NamedTarget:
    name: str
    target: Generator[Any, T, Any]


@dataclass(frozen=True)
class DeliveryFailure:
    target_name: str
    item_repr: str
    error: str


@coroutine
def resilient_broadcast_stage(
    targets: Sequence[NamedTarget],
    *,
    policy: str = "fail_fast",
    failure_target: Generator[Any, DeliveryFailure, Any] | None = None,
) -> Generator[None, T, None]:
    if policy not in {"fail_fast", "isolate"}:
        raise ValueError("policy must be 'fail_fast' or 'isolate'")

    active = list(targets)
    owned = [entry.target for entry in active]
    if failure_target is not None:
        owned.append(failure_target)

    try:
        while True:
            item = yield
            for entry in tuple(active):
                try:
                    entry.target.send(item)
                except Exception as exc:
                    failure = DeliveryFailure(entry.name, repr(item), f"{type(exc).__name__}: {exc}")
                    if failure_target is not None:
                        failure_target.send(failure)
                    if policy == "fail_fast":
                        raise
                    active.remove(entry)
    finally:
        close_unique(owned)


@coroutine
def exploding_sink(*, fail_on_vin: str) -> Generator[None, CarRecord, None]:
    while True:
        car = yield
        if car.vin == fail_on_vin:
            raise RuntimeError("simulated sink failure")

In [20]:
healthy_a: list[CarRecord] = []
healthy_b: list[CarRecord] = []
failures: list[DeliveryFailure] = []
fail_vin = valid_records[20].vin

resilient = resilient_broadcast_stage(
    [
        NamedTarget("healthy_a", append_sink(healthy_a)),
        NamedTarget("unstable", exploding_sink(fail_on_vin=fail_vin)),
        NamedTarget("healthy_b", append_sink(healthy_b)),
    ],
    policy="isolate",
    failure_target=append_sink(failures),
)

for car in valid_records[:50]:
    resilient.send(car)
resilient.close()

assert len(failures) == 1
assert failures[0].target_name == "unstable"
assert healthy_a == valid_records[:50]
assert healthy_b == valid_records[:50]
print(failures[0])

DeliveryFailure(target_name='unstable', item_repr="CarRecord(make='Ford', model='F-150', year=2013, vin='VIN00000000000020', color='Black')", error='RuntimeError: simulated sink failure')


## Problem 8 — Instrumentation and stage metrics

Create a generic instrumented stage. An operation may:

- return a transformed item,
- return a special `DROP` marker,
- raise an exception.

Measure received, emitted, dropped, failed, and elapsed time. Metrics should still be finalized when the stage is closed or fails.

### Solution 8

In [21]:
DROP = object()


@dataclass
class StageMetrics:
    name: str
    received: int = 0
    emitted: int = 0
    dropped: int = 0
    failed: int = 0
    elapsed_seconds: float = 0.0


@coroutine
def instrumented_stage(
    operation: Callable[[T], Any],
    target: Generator[Any, Any, Any],
    metrics: StageMetrics,
) -> Generator[None, T, None]:
    started = time.perf_counter()
    try:
        while True:
            item = yield
            metrics.received += 1
            try:
                result = operation(item)
            except Exception:
                metrics.failed += 1
                raise

            if result is DROP:
                metrics.dropped += 1
            else:
                target.send(result)
                metrics.emitted += 1
    finally:
        metrics.elapsed_seconds += time.perf_counter() - started
        target.close()

In [22]:
metric_output: list[CarRecord] = []
metrics = StageMetrics("keep_modern")
measured = instrumented_stage(
    lambda car: car if car.year >= 2000 else DROP,
    append_sink(metric_output),
    metrics,
)
for car in valid_records:
    measured.send(car)
measured.close()

assert metrics.received == len(valid_records)
assert metrics.emitted == len(metric_output)
assert metrics.dropped == len(valid_records) - len(metric_output)
assert metrics.failed == 0
assert metrics.elapsed_seconds >= 0
metrics

StageMetrics(name='keep_modern', received=240, emitted=103, dropped=137, failed=0, elapsed_seconds=0.0011983998119831085)

### Extra example — lightweight throughput calculation

Avoid over-interpreting tiny in-notebook benchmarks, but expose enough information for operational dashboards.

In [23]:
throughput = metrics.received / metrics.elapsed_seconds if metrics.elapsed_seconds else float("inf")
print(f"Approximate in-process throughput: {throughput:,.0f} records/second")

Approximate in-process throughput: 200,267 records/second


## Problem 9 — Atomic CSV sinks with rollback

Writing directly to a final file can leave partial output after a crash. Create a context-managed sink that:

1. Writes to a temporary file in the same directory.
2. Flushes and calls `fsync` on successful completion.
3. Atomically replaces the destination with `os.replace`.
4. Deletes the temporary file if the body raises.
5. Accepts either dataclass records or mappings.

This gives readers an all-old or all-new view of the destination, never a half-written file.

### Solution 9

In [24]:
def record_to_mapping(item: Any) -> Mapping[str, Any]:
    if hasattr(item, "__dataclass_fields__"):
        return asdict(item)
    if isinstance(item, Mapping):
        return item
    raise TypeError("CSV sink accepts dataclass instances or mappings")


@contextmanager
def atomic_csv_sink(
    final_path: Path,
    *,
    fieldnames: Sequence[str],
) -> Iterator[Generator[Any, Any, Any]]:
    final_path.parent.mkdir(parents=True, exist_ok=True)
    descriptor, temp_name = tempfile.mkstemp(
        dir=final_path.parent,
        prefix=f".{final_path.name}.",
        suffix=".tmp",
        text=True,
    )
    temp_path = Path(temp_name)
    file = os.fdopen(descriptor, "w", newline="", encoding="utf-8")
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()

    @coroutine
    def sink() -> Generator[None, Any, None]:
        while True:
            item = yield
            mapping = record_to_mapping(item)
            writer.writerow({name: mapping[name] for name in fieldnames})

    target = sink()
    try:
        yield target
    except BaseException:
        target.close()
        file.close()
        temp_path.unlink(missing_ok=True)
        raise
    else:
        target.close()
        file.flush()
        os.fsync(file.fileno())
        file.close()
        os.replace(temp_path, final_path)
    finally:
        if not file.closed:
            file.close()
        temp_path.unlink(missing_ok=True)

In [25]:
ATOMIC_PATH = WORKDIR / "atomic_cars.csv"
FIELDS = ["make", "model", "year", "vin", "color"]

with atomic_csv_sink(ATOMIC_PATH, fieldnames=FIELDS) as sink:
    for car in valid_records[:12]:
        sink.send(car)

with ATOMIC_PATH.open(newline="", encoding="utf-8") as file:
    committed_rows = list(csv.DictReader(file))
assert len(committed_rows) == 12

before_failure = ATOMIC_PATH.read_text(encoding="utf-8")
try:
    with atomic_csv_sink(ATOMIC_PATH, fieldnames=FIELDS) as sink:
        sink.send(valid_records[0])
        raise RuntimeError("simulate orchestration failure")
except RuntimeError:
    pass

after_failure = ATOMIC_PATH.read_text(encoding="utf-8")
assert after_failure == before_failure
print(f"Committed rows: {len(committed_rows)}; rollback preserved prior file: True")

Committed rows: 12; rollback preserved prior file: True


## Problem 10 — Safe configuration-driven pipeline construction

Create a pipeline from a plain dictionary while avoiding `eval`. Requirements:

- A registry maps approved predicate names to predicate factories.
- Unknown predicate names fail before processing starts.
- Each route writes atomically to its own CSV file.
- `ExitStack` manages a variable number of output context managers.
- The returned pipeline closes before the sinks commit.

### Solution 10

In [26]:
PredicateFactory = Callable[..., Callable[[CarRecord], bool]]


def color_is(*, color: str) -> Callable[[CarRecord], bool]:
    return lambda car: car.color.casefold() == color.casefold()


def make_and_color(*, make: str, color: str) -> Callable[[CarRecord], bool]:
    return lambda car: car.make.casefold() == make.casefold() and car.color.casefold() == color.casefold()


def year_before(*, year: int) -> Callable[[CarRecord], bool]:
    return lambda car: car.year < int(year)


def year_at_least(*, year: int) -> Callable[[CarRecord], bool]:
    return lambda car: car.year >= int(year)


PREDICATE_REGISTRY: dict[str, PredicateFactory] = {
    "color_is": color_is,
    "make_and_color": make_and_color,
    "year_before": year_before,
    "year_at_least": year_at_least,
}


@contextmanager
def configured_file_pipeline(
    config: Mapping[str, Any],
    *,
    output_dir: Path,
) -> Iterator[Generator[Any, CarRecord, Any]]:
    route_specs = config.get("routes", [])
    if not route_specs:
        raise ValueError("config must contain at least one route")

    # Validate every predicate before opening files.
    compiled: list[tuple[Mapping[str, Any], Callable[[CarRecord], bool]]] = []
    for spec in route_specs:
        predicate_name = spec["predicate"]
        try:
            factory = PREDICATE_REGISTRY[predicate_name]
        except KeyError as exc:
            raise ValueError(f"unknown predicate: {predicate_name}") from exc
        compiled.append((spec, factory(**spec.get("params", {}))))

    output_dir.mkdir(parents=True, exist_ok=True)
    with ExitStack() as stack:
        routes: list[Route] = []
        for spec, predicate in compiled:
            target = stack.enter_context(
                atomic_csv_sink(output_dir / spec["output"], fieldnames=FIELDS)
            )
            routes.append(Route(spec["name"], predicate, target))

        unmatched_target = None
        unmatched_name = config.get("unmatched_output")
        if unmatched_name:
            unmatched_target = stack.enter_context(
                atomic_csv_sink(output_dir / unmatched_name, fieldnames=FIELDS)
            )

        router = route_stage(
            routes,
            mode=config.get("mode", "all"),
            unmatched_target=unmatched_target,
        )
        try:
            yield router
        finally:
            router.close()

In [27]:
CONFIG = {
    "mode": "all",
    "routes": [
        {
            "name": "pink",
            "predicate": "color_is",
            "params": {"color": "Pink"},
            "output": "pink.csv",
        },
        {
            "name": "ford_green",
            "predicate": "make_and_color",
            "params": {"make": "Ford", "color": "Green"},
            "output": "ford_green.csv",
        },
        {
            "name": "older",
            "predicate": "year_before",
            "params": {"year": 1990},
            "output": "older.csv",
        },
    ],
    "unmatched_output": "unmatched.csv",
}

CONFIG_OUTPUT = WORKDIR / "configured_routes"
with configured_file_pipeline(CONFIG, output_dir=CONFIG_OUTPUT) as pipeline:
    for car in valid_records:
        pipeline.send(car)


def csv_row_count(path: Path) -> int:
    with path.open(newline="", encoding="utf-8") as file:
        return sum(1 for _ in csv.DictReader(file))


configured_counts = {
    path.name: csv_row_count(path)
    for path in sorted(CONFIG_OUTPUT.glob("*.csv"))
}

assert configured_counts["pink.csv"] == sum(car.color == "Pink" for car in valid_records)
assert configured_counts["ford_green.csv"] == sum(
    car.make == "Ford" and car.color == "Green" for car in valid_records
)
assert configured_counts["older.csv"] == sum(car.year < 1990 for car in valid_records)
configured_counts

{'ford_green.csv': 2, 'older.csv': 102, 'pink.csv': 34, 'unmatched.csv': 118}

### Extra example — reject unsafe or unknown configuration before creating output

In [28]:
unsafe_config = {
    "routes": [
        {
            "name": "bad",
            "predicate": "__import__('os').system",
            "params": {},
            "output": "bad.csv",
        }
    ]
}

try:
    with configured_file_pipeline(unsafe_config, output_dir=WORKDIR / "should_not_exist"):
        pass
except ValueError as exc:
    print(exc)
else:
    raise AssertionError("unknown predicates must be rejected")

unknown predicate: __import__('os').system


## Problem 11 — Asynchronous broadcasting with backpressure

Generator coroutines run synchronously: a slow target blocks the producer. Build an `asyncio` version that:

- gives each consumer its own bounded queue,
- awaits `queue.put`, creating backpressure when a consumer falls behind,
- sends a sentinel to terminate consumers,
- allows consumers to filter independently,
- returns each consumer's accepted records.

A bounded queue is essential; an unbounded queue can turn a slow consumer into unbounded memory growth.

### Solution 11

In [29]:
ASYNC_END = object()


async def async_producer(
    items: Iterable[CarRecord],
    queues: Sequence[asyncio.Queue[Any]],
) -> None:
    for item in items:
        for queue in queues:
            await queue.put(item)
    for queue in queues:
        await queue.put(ASYNC_END)


async def async_consumer(
    name: str,
    queue: asyncio.Queue[Any],
    predicate: Callable[[CarRecord], bool],
    *,
    artificial_delay: float = 0.0,
) -> tuple[str, list[CarRecord]]:
    accepted: list[CarRecord] = []
    while True:
        item = await queue.get()
        try:
            if item is ASYNC_END:
                return name, accepted
            if predicate(item):
                accepted.append(item)
            if artificial_delay:
                await asyncio.sleep(artificial_delay)
        finally:
            queue.task_done()


async def async_broadcast_demo(records: Sequence[CarRecord]) -> dict[str, list[CarRecord]]:
    pink_queue: asyncio.Queue[Any] = asyncio.Queue(maxsize=8)
    modern_queue: asyncio.Queue[Any] = asyncio.Queue(maxsize=8)

    consumers = [
        asyncio.create_task(async_consumer("pink", pink_queue, lambda car: car.color == "Pink")),
        asyncio.create_task(
            async_consumer(
                "modern",
                modern_queue,
                lambda car: car.year >= 2015,
                artificial_delay=0.0001,
            )
        ),
    ]

    await async_producer(records, [pink_queue, modern_queue])
    results = await asyncio.gather(*consumers)
    return dict(results)

In [30]:
async_results = await async_broadcast_demo(valid_records[:100])

assert len(async_results["pink"]) == sum(car.color == "Pink" for car in valid_records[:100])
assert len(async_results["modern"]) == sum(car.year >= 2015 for car in valid_records[:100])
assert all(car.color == "Pink" for car in async_results["pink"])
assert all(car.year >= 2015 for car in async_results["modern"])

{name: len(records) for name, records in async_results.items()}

{'pink': 15, 'modern': 21}

## Capstone — Validated, deduplicated, measured, multi-route pipeline

Build one integrated in-memory pipeline that:

1. Streams and validates the CSV.
2. Sends parse issues to a dead-letter list.
3. Deduplicates VINs with bounded state.
4. Measures an identity/normalization stage.
5. Routes records to every matching business rule.
6. Captures unmatched records.
7. Produces a JSON summary and verifies every invariant.

### Capstone solution

In [31]:
capstone_issues: list[ParseIssue] = []
capstone_issue_sink = append_sink(capstone_issues)

capstone_pink: list[CarRecord] = []
capstone_ford_green: list[CarRecord] = []
capstone_old: list[CarRecord] = []
capstone_unmatched: list[CarRecord] = []

capstone_router = route_stage(
    [
        Route("pink", lambda car: car.color == "Pink", append_sink(capstone_pink)),
        Route(
            "ford_green",
            lambda car: car.make == "Ford" and car.color == "Green",
            append_sink(capstone_ford_green),
        ),
        Route("old", lambda car: car.year < 1990, append_sink(capstone_old)),
    ],
    mode="all",
    unmatched_target=append_sink(capstone_unmatched),
)

capstone_metrics = StageMetrics("normalize")
measured_normalizer = instrumented_stage(normalize_car, capstone_router, capstone_metrics)
capstone_dedupe_stats = DedupeStats()
capstone_pipeline = dedupe_stage(
    key=lambda car: car.vin,
    target=measured_normalizer,
    max_seen=500,
    stats=capstone_dedupe_stats,
)

for car in stream_car_records(DATA_PATH, error_target=capstone_issue_sink):
    capstone_pipeline.send(car)

capstone_pipeline.close()
capstone_issue_sink.close()

unique_valid = list(dict.fromkeys(car.vin for car in valid_records))
expected_unique_count = len(unique_valid)

assert len(capstone_issues) == 4
assert capstone_dedupe_stats.received == len(valid_records)
assert capstone_dedupe_stats.emitted == expected_unique_count
assert capstone_metrics.received == expected_unique_count
assert capstone_metrics.emitted == expected_unique_count
assert all(car.color == "Pink" for car in capstone_pink)
assert all(car.make == "Ford" and car.color == "Green" for car in capstone_ford_green)
assert all(car.year < 1990 for car in capstone_old)
assert all(
    not (car.color == "Pink" or (car.make == "Ford" and car.color == "Green") or car.year < 1990)
    for car in capstone_unmatched
)

capstone_summary = {
    "input_rows_including_invalid": 244,
    "valid_rows": len(valid_records),
    "parse_issues": len(capstone_issues),
    "unique_valid_vins": expected_unique_count,
    "duplicates_dropped": capstone_dedupe_stats.dropped,
    "routes": {
        "pink": len(capstone_pink),
        "ford_green": len(capstone_ford_green),
        "older_than_1990": len(capstone_old),
        "unmatched": len(capstone_unmatched),
    },
    "metrics": asdict(capstone_metrics),
}

SUMMARY_PATH = WORKDIR / "capstone_summary.json"
SUMMARY_PATH.write_text(json.dumps(capstone_summary, indent=2), encoding="utf-8")
capstone_summary

{'input_rows_including_invalid': 244,
 'valid_rows': 240,
 'parse_issues': 4,
 'unique_valid_vins': 235,
 'duplicates_dropped': 5,
 'routes': {'pink': 33,
  'ford_green': 2,
  'older_than_1990': 100,
  'unmatched': 115},
 'metrics': {'name': 'normalize',
  'received': 235,
  'emitted': 235,
  'dropped': 0,
  'failed': 0,
  'elapsed_seconds': 0.006226199679076672}}

### Capstone verification — independent count checks

Independent computations reduce the chance that the test merely repeats the implementation's mistake.

In [32]:
unique_by_vin: dict[str, CarRecord] = {}
for car in valid_records:
    unique_by_vin.setdefault(car.vin, car)
independent = list(unique_by_vin.values())

expected_routes = {
    "pink": sum(car.color == "Pink" for car in independent),
    "ford_green": sum(car.make == "Ford" and car.color == "Green" for car in independent),
    "older_than_1990": sum(car.year < 1990 for car in independent),
    "unmatched": sum(
        not (
            car.color == "Pink"
            or (car.make == "Ford" and car.color == "Green")
            or car.year < 1990
        )
        for car in independent
    ),
}

assert capstone_summary["routes"] == expected_routes
print("All capstone invariants passed.")

All capstone invariants passed.


## Additional solved micro-examples

These compact examples add more lines of reusable code around inspection, composition, and defensive behavior.

### Micro-example A — reusable predicate combinators

In [33]:
def all_of(*predicates: Callable[[T], bool]) -> Callable[[T], bool]:
    return lambda item: all(predicate(item) for predicate in predicates)


def any_of(*predicates: Callable[[T], bool]) -> Callable[[T], bool]:
    return lambda item: any(predicate(item) for predicate in predicates)


def negate(predicate: Callable[[T], bool]) -> Callable[[T], bool]:
    return lambda item: not predicate(item)


is_pink = lambda car: car.color == "Pink"
is_ford = lambda car: car.make == "Ford"
is_classic = lambda car: car.year < 1990

pink_fords = [car for car in valid_records if all_of(is_pink, is_ford)(car)]
modern_non_pink = [car for car in valid_records if all_of(negate(is_classic), negate(is_pink))(car)]
assert all(car.make == "Ford" and car.color == "Pink" for car in pink_fords)
assert all(car.year >= 1990 and car.color != "Pink" for car in modern_non_pink)
len(pink_fords), len(modern_non_pink)

(3, 118)

### Micro-example B — sampling stage for debugging without flooding output

In [34]:
@coroutine
def sample_stage(
    every: int,
    observer: Callable[[T], None],
    target: Generator[Any, T, Any],
) -> Generator[None, T, None]:
    if every <= 0:
        raise ValueError("every must be positive")
    count = 0
    try:
        while True:
            item = yield
            count += 1
            if count % every == 0:
                observer(item)
            target.send(item)
    finally:
        target.close()


sampled_vins: list[str] = []
forwarded: list[CarRecord] = []
sampler = sample_stage(25, lambda car: sampled_vins.append(car.vin), append_sink(forwarded))
for car in valid_records[:100]:
    sampler.send(car)
sampler.close()

assert len(sampled_vins) == 4
assert forwarded == valid_records[:100]
sampled_vins

['VIN00000000000024',
 'VIN00000000000049',
 'VIN00000000000074',
 'VIN00000000000099']

### Micro-example C — flatten batches back into individual items

In [35]:
@coroutine
def flatten_stage(target: Generator[Any, T, Any]) -> Generator[None, Iterable[T], None]:
    try:
        while True:
            items = yield
            for item in items:
                target.send(item)
    finally:
        target.close()


round_trip: list[CarRecord] = []
round_trip_pipeline = batch_stage(17, flatten_stage(append_sink(round_trip)))
for car in valid_records[:53]:
    round_trip_pipeline.send(car)
round_trip_pipeline.close()
assert round_trip == valid_records[:53]
len(round_trip)

53

## Additional practice problems

Try these without looking back at the solutions above.

1. **Sliding-window average:** maintain the average model year over the most recent 25 records using `collections.deque(maxlen=25)`.
2. **Rate limiter:** permit at most `N` items per second while preserving order. Make the clock and sleep functions injectable for deterministic tests.
3. **Retrying sink:** retry selected exception types with exponential backoff and a maximum attempt count. Route permanently failed items to a dead-letter sink.
4. **Partitioned ordering:** route by make, preserve order within each make, and process different makes concurrently.
5. **Checkpointing:** persist the last committed row number and resume safely after interruption without duplicating already committed output.
6. **Schema evolution:** accept both `model_year` and `year` input columns while producing one canonical `CarRecord` schema.
7. **Watermarks:** for timestamped records, detect excessively late data and route it to a late-events sink.
8. **Circuit breaker:** stop sending to a repeatedly failing target for a cooldown period, then allow a probe item.
9. **Hash partitioning:** route VINs across `k` output shards while guaranteeing the same VIN always goes to the same shard.
10. **Exactly-once discussion:** explain why a local generator pipeline alone cannot guarantee distributed exactly-once delivery, then identify the storage-level mechanisms needed.

## Design checklist

Before shipping a pipeline, verify:

- **Boundaries:** input schema, encoding, validation, and malformed-row policy are explicit.
- **Ownership:** every opened file, queue, task, and coroutine has one clear owner.
- **Shutdown:** downstream stages flush buffered data and close in a predictable order.
- **State:** dedupe caches, windows, and queues have bounded memory.
- **Failure policy:** fail-fast, retry, isolate, rollback, and dead-letter behavior are intentional.
- **Observability:** counts, errors, latency, throughput, and route volumes are measurable.
- **Configuration:** only registered operations are allowed; never execute arbitrary expressions.
- **Testing:** normal, empty, malformed, duplicate, overlapping-route, partial-batch, and failure cases are covered.
- **Persistence:** critical files use atomic replacement or transactional storage.
- **Concurrency:** bounded queues enforce backpressure and task cancellation is handled.

## Final notes

The synchronous coroutine approach is excellent for teaching composition and for single-process streaming tasks. For production systems, choose the simplest abstraction that satisfies requirements:

- regular iterators for pull-based transformations,
- generator coroutines for synchronous push pipelines,
- `asyncio` queues for cooperative concurrency and backpressure,
- process pools for CPU-bound parallelism,
- a durable stream processor or message broker when replay, partitioning, distributed recovery, and cross-machine guarantees are required.